# Python Decorators: Practical Syntax Sugar

A comprehensive guide to understanding what truly happens behind the `@decorator` syntax.

## Objectives

By the end of this lesson, you will be able to:

- Explain decorators using first-class functions;
- Manually expand the `@` syntax;
- Create simple decorators and decorators that accept arguments;
- Preserve function metadata with `functools.wraps`;
- Chain and stack multiple decorators in the correct order;
- Avoid common pitfalls with arguments, return values, and methods.

> **Core Idea:** A decorator receives a function, modifies or extends its behavior, and returns a callable bound to the original name.

In [1]:
from functools import wraps


def log_call(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Executing: {func.__name__}")
        result = func(*args, **kwargs)
        print(f"Result: {result}")
        return result

    return wrapper


@log_call
def add(a, b):
    """Adds two numbers together."""
    return a + b


assert add(2, 3) == 5
assert add.__name__ == "add"
assert add.__doc__ == "Adds two numbers together."


def repeat(num_times):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            result = None
            for _ in range(num_times):
                result = func(*args, **kwargs)
            return result

        return wrapper

    return decorator


call_log = []


@repeat(3)
def register_user(name):
    call_log.append(name)
    return name


assert register_user("Alice") == "Alice"
assert call_log == ["Alice", "Alice", "Alice"]
print("All basic decorator tests passed successfully.")

Executing: add
Result: 5
All basic decorator tests passed successfully.


## Understanding the `@` Syntax

In Python, writing:

```python
@log_call
def add(a, b):
    return a + b
```

is pure syntax sugar for:

```python
def add(a, b):
    return a + b

add = log_call(add)
```

The replacement occurs when the function definition is executed at load time. The wrapper then executes on every function call.

In [2]:
def time_execution(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        import time

        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        duration = time.perf_counter() - start_time
        print(f"{func.__name__} took {duration:.6f}s")
        return result

    return wrapper


@time_execution
def generate_squares(limit):
    return [num ** 2 for num in range(limit)]


print(generate_squares(5))

generate_squares took 0.000002s
[0, 1, 4, 9, 16]


## Decorators with Arguments

Decorators that take arguments introduce three nested levels:

```python
@repeat(3)
def say_hello():
    ...
```

1. `repeat(3)` receives the configuration argument and returns a decorator;
2. That decorator takes the target function as input;
3. The wrapper handles the arguments when the function is called.

The expanded form is `say_hello = repeat(3)(say_hello)`.

In [3]:
@repeat(3)
def say_hello(name):
    print(f"Hello, {name}!")


say_hello("Alice")

Hello, Alice!
Hello, Alice!
Hello, Alice!


## `functools.wraps`: Preserving Identity

A wrapper function replaces the original function, which can overwrite `__name__` and `__doc__`. Using `@wraps(func)` copies over this metadata, ensuring documentation tools, debuggers, and introspection work seamlessly.

Without `wraps`, introspection tools will show `wrapper` as the function name instead of the original name.

In [4]:
@log_call
def rectangle_area(width, height):
    """Calculates the area of a rectangle."""
    return width * height


print(rectangle_area(3, 4))
print(rectangle_area.__name__)
print(rectangle_area.__doc__)

Executing: rectangle_area
Result: 12
12
rectangle_area
Calculates the area of a rectangle.


## Stacking Multiple Decorators: Order Matters

```python
@A
@B
def function():
    pass
```

is equivalent to `function = A(B(function))`. The decorator closest to the function (`@B`) is applied first. Keep this rule in mind when predicting execution flow and output order.

In [5]:
def trace(label):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            print(f"Entering {label}")
            result = func(*args, **kwargs)
            print(f"Exiting {label}")
            return result

        return wrapper

    return decorator


@trace("outer")
@trace("inner")
def greet():
    print("Function body execution")


greet()

Entering outer
Entering inner
Function body execution
Exiting inner
Exiting outer


## Decorators on Class Methods

When decorating methods, `self` is simply the first positional argument passed to the wrapper. Decorators can inspect or validate the object's internal state before invoking the original method.

In [6]:
def require_active(func):
    @wraps(func)
    def wrapper(self, *args, **kwargs):
        if not self.is_active:
            raise RuntimeError("User is inactive.")
        return func(self, *args, **kwargs)

    return wrapper


class User:
    def __init__(self, name, is_active=True):
        self.name = name
        self.is_active = is_active

    @require_active
    def access_dashboard(self):
        return f"Dashboard of {self.name}"


print(User("Alice").access_dashboard())
try:
    User("Bob", is_active=False).access_dashboard()
except RuntimeError as err:
    print(type(err).__name__, err)

Dashboard of Alice
RuntimeError User is inactive.


## Common Pitfalls & Practical Exercise

Best practices to remember:

- Always return the result of the wrapped function;
- Forward `*args` and `**kwargs` to accommodate arbitrary arguments;
- Always use `@wraps(func)`;
- Check the evaluation order when stacking multiple decorators;
- Never silently swallow exceptions without an explicit reason.

### Exercise

Implement a decorator `require_token` that only permits execution if a keyword argument `token` equals `"python"`. See the solution in the next cell.

In [7]:
def require_token(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        if kwargs.get("token") != "python":
            raise PermissionError("Invalid token.")
        return func(*args, **kwargs)

    return wrapper


@require_token
def secret_data(*, token):
    return "Confidential access granted."


assert secret_data(token="python") == "Confidential access granted."
try:
    secret_data(token="wrong_token")
except PermissionError as err:
    print(type(err).__name__, err)

PermissionError Invalid token.


## Summary

```python
@decorator
def function(...):
    ...
```

is syntactic sugar for:

```python
def function(...):
    ...

function = decorator(function)
```

Decorators combine function composition, closures, and variable reassignment. They are an elegant Pythonic pattern for extending behavior cleanly without modifying the core function logic.